# Kaggle Run From Git

Thin Kaggle notebook: clone or pull the latest repo code, then run the Python entrypoint.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/BDT-17/Internship.git"
REPO_DIR = Path("/kaggle/working/Internship")
BRANCH = None

# --- Run settings ---
SMOKE = True            # True -> quick 1-epoch check before the real run
EPOCHS = 50
BATCH_SIZE = 16
MODELS = ["custom_cnn_v2"]

# --- Tier-1 training options (defaults reproduce the old behaviour) ---
OPTIMIZER = "adamw"              # adam | adamw
LR_SCHEDULER = "cosine_warmup"   # none | cosine | cosine_warmup
LABEL_SMOOTHING = 0.1
LOSS = "cb_focal"                # ce | focal | class_balanced | cb_focal
MIXUP_ALPHA = 0.2
CUTMIX_ALPHA = 1.0


In [ ]:
import subprocess

if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "fetch", "--all"], cwd=REPO_DIR, check=True)
    if BRANCH:
        subprocess.run(["git", "checkout", BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "pull", "--ff-only"], cwd=REPO_DIR, check=True)
else:
    command = ["git", "clone"]
    if BRANCH:
        command.extend(["--branch", BRANCH])
    command.extend([REPO_URL, str(REPO_DIR)])
    subprocess.run(command, check=True)

print("Repository ready:", REPO_DIR)

In [ ]:
%cd /kaggle/working/Internship
import subprocess

epochs = 1 if SMOKE else EPOCHS
cmd = [
    "python", "scripts/train_kaggle.py",
    "--epochs", str(epochs),
    "--batch-size", str(BATCH_SIZE),
    "--models", *MODELS,
    "--optimizer", OPTIMIZER,
    "--lr-scheduler", LR_SCHEDULER,
    "--label-smoothing", str(LABEL_SMOOTHING),
    "--loss", LOSS,
    "--mixup-alpha", str(MIXUP_ALPHA),
    "--cutmix-alpha", str(CUTMIX_ALPHA),
]
if SMOKE:
    cmd.append("--no-save-every-epoch")
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
from pathlib import Path

output_root = Path("/kaggle/working/plant_training_outputs")
for path in sorted(output_root.rglob("*")):
    if path.is_file():
        print(path.relative_to(output_root))

## Training curves

Overlay the validation curves and print a per-model convergence table.

In [ ]:
# --- Training curves: overlay validation loss/accuracy for every trained model ---
import sys
from pathlib import Path

sys.path.insert(0, "/kaggle/working/Internship")
from plant_classifier.analysis import save_convergence_comparison

output_root = Path("/kaggle/working/plant_training_outputs")
curve_models = tuple(
    p.name
    for p in sorted(output_root.iterdir())
    if p.is_dir() and (p / "history.csv").exists()
)
if curve_models:
    table = save_convergence_comparison(
        output_root, curve_models, output_root / "training_curves.png"
    )
    print("Wrote", output_root / "training_curves.png")
    print("Per-model convergence (best epoch, val acc, overfit gap):")
    print(table.to_string(index=False))
else:
    print("No history.csv found under", output_root, "- run the training cell first.")


## Download bundle

Zip the figures + metrics needed for the report into one archive.

In [ ]:
# --- Bundle the outputs needed to write up the research, ready to download ---
# Zips every report artifact (figures + CSV/JSON metrics) from the training and
# analysis output dirs into one archive in /kaggle/working. Per-epoch checkpoints
# are always excluded; best_model.pth is excluded unless BUNDLE_WEIGHTS = True.
import zipfile
from pathlib import Path

BUNDLE_WEIGHTS = False  # True -> also include best_model.pth for each model (large)

SOURCE_DIRS = [
    Path("/kaggle/working/plant_training_outputs"),  # training: summaries, curves, per-model figs
    Path("/kaggle/working/analysis"),                # analyze_kaggle.py: per-class / source / confusion
]
BUNDLE_PATH = Path("/kaggle/working/plant_research_bundle.zip")


def keep(path: Path) -> bool:
    name = path.name
    if name.startswith("epoch_") and name.endswith(".pth"):
        return False  # per-epoch checkpoints are never needed for the writeup
    if name == "best_model.pth" and not BUNDLE_WEIGHTS:
        return False
    return True


count = 0
with zipfile.ZipFile(BUNDLE_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for root in SOURCE_DIRS:
        if not root.exists():
            continue
        for path in sorted(root.rglob("*")):
            if path.is_file() and keep(path):
                zf.write(path, path.relative_to(root.parent))  # keep dir name in the zip
                count += 1

if count:
    size_mb = BUNDLE_PATH.stat().st_size / 1e6
    print(f"Wrote {BUNDLE_PATH} ({count} files, {size_mb:.1f} MB)")
    print("Download it from the Kaggle output panel (Data > /kaggle/working) on the right.")
else:
    print("Nothing to bundle - run training (and optionally analyze_kaggle.py) first.")
